### 3-Shot Learning using LLM and Clustering ###

the purpose of this notebook, is to is to use contextual embeddings, bertopic pipiline from the previous experiment and adapt it for kmeans.


LLM-based interpretation: the LLM converts an unlabeled numerical cluster into a meaningful topic.

Imports/Installs

In [50]:
# # Installs Unsloth, Xformers (Flash Attention)
# !pip install unsloth
# # Get latest Unsloth
# !pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [51]:
# %pip install -q bertopic sentence-transformers umap-learn

In [28]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from unsloth import FastLanguageModel
from transformers import TextStreamer
from unsloth.chat_templates import get_chat_template
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from umap import UMAP
import re

In [29]:
lama_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "ilsp/Llama-Krikri-8B-Instruct", max_seq_length = 8192, load_in_4bit = True,)

==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

read dataframe

In [30]:
df = pd.read_csv("hf://datasets/DominusTea/GreekLegalSum/hugginface_dataset.csv")

COntextual embeddings will be applied to summaries instead of text.

In [31]:
# docs = (df["summary"].dropna().astype(str).loc[lambda x: x.str.strip().ne("")].tolist())

In [32]:
valid_rows = (
    df["summary"].notna() &
    df["summary"].astype(str).str.strip().ne("") &
    df["case_category"].notna() &
    df["case_category"].astype(str).str.strip().ne("")
)

filtered_df = df.loc[valid_rows].copy()

docs = filtered_df["summary"].astype(str).tolist()
titles = filtered_df["case_category"].astype(str).tolist()


In [33]:
# Bertopic pipeline with kmeans
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

kmeans_model = KMeans(n_clusters=5, n_init=20, random_state=42)

# Reduction of contextual embeddings before K-Means using umap
umap_model = UMAP(n_neighbors=15,n_components=5,min_dist=0.0,metric="cosine",random_state=42)

vectorizer_model = CountVectorizer(min_df=2, ngram_range=(1, 2))

# BERTopic pipeline with K-Means
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,

    # kmeans instead
    hdbscan_model=kmeans_model,

    vectorizer_model=vectorizer_model,
    top_n_words=10,
    calculate_probabilities=False,
    low_memory=True,
    verbose=True
)


# probabilities not relevant in kmens
topics, _ = topic_model.fit_transform(docs)


# topic_info = topic_model.get_topic_info()

# display(topic_info[["Topic", "Count", "Name", "Representation"]])

fitted_kmeans = topic_model.hdbscan_model

# Raw K-Means labels
labels = fitted_kmeans.labels_

# Cluster centroids
centroids = fitted_kmeans.cluster_centers_

# reduced_embeddings[i] → document vector
doc_embedding = topic_model.umap_model.embedding_

print("Labels shape:", labels.shape)
print("Centroids shape:", centroids.shape)
print("Reduced embeddings shape:", doc_embedding.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-09-05 18:08:45,593 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/197 [00:00<?, ?it/s]

2026-09-05 18:09:02,712 - BERTopic - Embedding - Completed ✓
2026-09-05 18:09:02,715 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-05 18:09:16,369 - BERTopic - Dimensionality - Completed ✓
2026-09-05 18:09:16,371 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-05 18:09:16,665 - BERTopic - Cluster - Completed ✓
2026-09-05 18:09:16,671 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-05 18:09:18,045 - BERTopic - Representation - Completed ✓


Labels shape: (6298,)
Centroids shape: (5, 5)
Reduced embeddings shape: (6298, 5)


In [34]:
silhouette_val = silhouette_score(
    doc_embedding,
    labels,
    metric="euclidean",
    sample_size=min(10_000, len(labels)),
    random_state=42
)

print(f"K-Means silhouette score: {silhouette_val:.4f}")

K-Means silhouette score: 0.2984


Capturing centroids and distances in a dataframe

In [35]:
# 8394 rows -> 8394 legal summaries;
# 5 columns -> five UMAP coordinates representing each summary.
# (8394, 5)
assigned_centroids = centroids[labels]
distances_to_centroid = np.linalg.norm( doc_embedding - assigned_centroids, axis=1)# Axis 1 runs horizontally across columns

In [36]:
# valid_rows = (
#     df["summary"].notna() &
#     df["summary"].astype(str).str.strip().ne("") &
#     df["case_category"].notna() &
#     df["case_category"].astype(str).str.strip().ne("")
# )

# filtered_df = df.loc[valid_rows].copy()

# docs = filtered_df["summary"].astype(str).tolist()
# titles = filtered_df["case_category"].astype(str).tolist()


In [37]:
# doc_idx connects each row in cluster_df to its original row position in df.
# doc_idx connects each row in cluster_df to its original row position in df.

cluster_df = pd.DataFrame({
    "doc_idx": np.arange(len(docs)),
    "text": docs,
    "cluster": labels,
    "distance_to_centroid": distances_to_centroid,
    "title" : titles
})

cluster_df.head()

,doc_idx,text,cluster,distance_to_centroid,title
0,0,Αίτηση αναίρεσης καταδικαστικής αποφάσεως για ...,0,1.101082,Ακυρότητα απόλυτη
1,1,Καθορισμός συνολικής ποινής (άρθρο 551 ΚΠΔ) με...,2,1.241688,Ποινή συνολική
2,2,Καταδικαστική απόφαση για μη καταβολή χρεών πρ...,2,1.238881,Ακυρότητα απόλυτη
3,3,Αναίρεση Εισαγγελέα Αρείου Πάγου κατά αθωωτική...,0,1.653784,Αβάσιμοι λόγοι
4,4,Κατ' εξακολούθηση απάτη κατ' επάγγελμα και κατ...,0,2.002539,Αβάσιμοι λόγοι


3 shot technique will be used on random sample, meaninng a title will be generated for 3 titles

In [38]:
random_samples = []

closest_centroids = (cluster_df.sort_values(["cluster", "distance_to_centroid"]).groupby("cluster", group_keys=False).head(3).copy())
display(closest_centroids)
random_points = (cluster_df.groupby("cluster", group_keys=False).sample(n=3, random_state=42).copy())
display(random_points)

,doc_idx,text,cluster,distance_to_centroid,title
4223,4223,Αναίρεση καταδικαστικής αποφάσεως για ψευδή κα...,0,0.194408,Αιτιολογίας ανεπάρκεια
1033,1033,Καταδικαστική απόφαση για ψευδή καταμήνυση και...,0,0.285553,Αιτιολογίας ανεπάρκεια
5791,5791,Αποδοχή προϊόντων εγκλήματος. Αναίρεση καταδικ...,0,0.309498,Αιτιολογίας ανεπάρκεια
2351,2351,Αίτηση αναιρέσεως κατά αποφάσεως Τριμελούς Πλη...,1,0.439537,Αιτιολογίας ανεπάρκεια
2079,2079,Παράβαση άρθρου 31 § 4 Αγορανομικού Κώδικα (ν....,1,0.448516,Αιτιολογίας ανεπάρκεια
228,228,Καταδικαστική απόφαση για υπεξαίρεση στην υπη...,1,0.501488,Υπέρβαση εξουσίας
1951,1951,Καταδικαστική απόφαση για παράβαση του νόμου π...,2,0.179865,Ακυρότητα απόλυτη
2566,2566,Διάθεση στην κατανάλωση τροφίμων ακαταλλήλων π...,2,0.227013,Ακυρότητα απόλυτη
4197,4197,Φαρμακοδιέγερση. Επικίνδυνη σωματική βλάβη. Υπ...,2,0.229237,Ακυρότητα απόλυτη
1367,1367,"Κανονισμός Αρμοδιότητας, άρθρο 136 ΚΠΔ. Συντρέ...",3,0.224336,Κανονισμός αρμοδιότητας


,doc_idx,text,cluster,distance_to_centroid,title
1480,1480,"Παραπεμπτικό βούλευμα για ψευδή καταμήνυση, ψε...",0,1.111846,Αιτιολογίας ανεπάρκεια
4205,4205,Απάτη. Ζημία από έλλειψη συνομολογηθείσας ιδιό...,0,0.935168,Ακυρότητα απόλυτη
1660,1660,Απόφαση καθορισμού συνολικής ποινής. Λόγος ανα...,0,0.756889,Ποινή συνολική
2643,2643,Χρήση πλαστού εγγράφου (Διδακτορικού τίτλου) κ...,1,1.467191,Ακροάσεως έλλειψη
3603,3603,Φθορά ξένης ιδιοκτησίας - άρθρ. 381 παρ. 1 ΠΚ....,1,1.329417,Αιτιολογίας επάρκεια
5933,5933,Ασφαλιστικές εισφορές Α.Ν. 86/1967. Δεν είναι ...,1,1.080415,Νόμου εφαρμογή και ερμηνεία
6048,6048,Αναιρείται η καταδικαστική από-φαση με την επί...,2,0.518450,Ακροάσεως Αρχή
5014,5014,Πότε υπάρχει αιτιολογία στην απόφαση με την οπ...,2,1.560191,Αιτιολογίας επάρκεια
6285,6285,Έννοια καταδικαστικής αποφάσεως. Νομική φύση τ...,2,1.039079,Ποινή
1408,1408,Βούλευμα. Κανονισμός Αρμοδιότητας. Αιτών: Ο Ει...,3,0.791770,Κανονισμός αρμοδιότητας


Examining the difference between titles generated from random samples and samples near centroid.

In [39]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",

)
FastLanguageModel.for_inference(lama_model) # Enable native 2x faster inference

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(149248, 4096, padding_idx=128255)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

In [40]:
def create_prompt(texts):
    if len(texts) != 3:
        raise ValueError("The prompt must contain exactly 3 texts.")

    decisions = "\n\n".join(
        f"Απόφαση {i}:\n{text}"
        for i, text in enumerate(texts, start=1)
    )

    return f"""Σου δίνονται τρεις περιλήψεις νομικών αποφάσεων που ανήκουν στην ίδια συστάδα.

Εντόπισε το κοινό κεντρικό νομικό θέμα τους και δημιούργησε
έναν σύντομο και συγκεκριμένο τίτλο στα ελληνικά.

Απάντησε αποκλειστικά στη μορφή:
Θέμα: <τίτλος>

{decisions}
""".strip()

In [41]:
def get_llm_output(prompt):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors="pt").to("cuda")

    outputs = lama_model.generate(input_ids=inputs, max_new_tokens=60, do_sample=False)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [42]:
def extract_title(llm_output):
    match = re.search(r"(?im)^\sΘέμα:\s*(.*)", llm_output, re.IGNORECASE | re.DOTALL)
    # (?im) -> i means case-insensitive matching.
    # m means multiline mode.

    if match:
        return match.group(1).strip()
    else:
        return "nothing"


In [43]:
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

def run_llm(df):
    titles = {}

    for i in df["cluster"].unique():

        print(f"\nlearning for cluster {i} :")

        rows_by_cluster = df.loc[df["cluster"] == i]

        texts = rows_by_cluster["text"].to_numpy()

        try:
            prompt = create_prompt(texts)

        except Exception as e:
          print("failed to create prompt")
          continue

        try:
            output = get_llm_output(prompt)

        except Exception as e:
            print("failed to get llm output")
            continue

        try:

            title = extract_title(output)

        except Exception as e:
            print("failed to exctarct title")
            continue

        titles[i] = title
        print("\nExtracted title with regex:\n", title)
    return titles


In [44]:
print("Running 3-shot Learning for 3 closest data points...")
cluster_results = run_llm(closest_centroids)
print("Running 3-shot Learning for 3 random data points...")
random_results = run_llm(random_points)

Running 3-shot Learning for 3 closest data points...

learning for cluster 0 :

Extracted title with regex:
 **Έλλειψη ειδικής και εμπεριστατωμένης αιτιολογίας σε ποινικές αποφάσεις**

Οι τρεις αποφάσεις μοιράζονται το κεντρικό νομικό ζήτημα της ανεπαρκούς ή ελλείπουσας ειδικής και εμπεριστατωμένης αιτιολογίας σε καταδικαστικές αποφάσεις ποινικών υποθέσεων.

learning for cluster 1 :

Extracted title with regex:
 Ευθύνη για αγορανομικές παραβάσεις και ερμηνεία ποινικής νομοθεσίας σχετικά με την ευθύνη νομικών προσώπων και υπαλλήλων

learning for cluster 2 :

Extracted title with regex:
 **Παραβιάσεις Διαδικαστικών Εγγυήσεων στην Ποινική Δίκη**

learning for cluster 3 :

Extracted title with regex:
 **Καθορισμός Αρμοδιότητας σε Εγκλήσεις κατά Δικαστικών Λειτουργών**

Οι τρεις αποφάσεις αφορούν τον κανονισμό αρμοδιότητας σε περιπτώσεις έγκλησης κατά δικαστικών λειτουργών, όπου η αρμοδιότητα μετατίθεται σε άλλες δικαστικές αρχές, ακόμη και κατά το στάδιο της προδικασίας, για λόγους διασφάλ

In [45]:
top_5_titles = (cluster_df.copy().groupby("cluster")["title"]

                .value_counts(normalize=True).mul(100).rename("percentage").reset_index().sort_values(["cluster", "percentage"], ascending=[True, False]

    ).groupby("cluster").head(5))

top_5_titles

,cluster,title,percentage
0,0,Αιτιολογίας επάρκεια,41.536204
1,0,Αιτιολογίας ανεπάρκεια,19.569472
2,0,Ακυρότητα απόλυτη,16.829746
3,0,Ακροάσεως έλλειψη,2.299413
4,0,Αοριστία λόγου αναιρέσεως,0.978474
157,1,Αιτιολογίας επάρκεια,21.126761
158,1,Αιτιολογίας ανεπάρκεια,12.419974
159,1,Ακυρότητα απόλυτη,10.051216
160,1,Έκδοση,4.481434
161,1,Χρησικτησία,4.033291


In [46]:
#items() method returns a view object

cluster_results_df = pd.DataFrame(cluster_results.items(),columns=["cluster", "centroid_title"])
display(cluster_results_df)

random_results_df = pd.DataFrame(random_results.items(),columns=["cluster", "random_title"])
display(random_results_df)

,cluster,centroid_title
0,0,**Έλλειψη ειδικής και εμπεριστατωμένης αιτιολο...
1,1,Ευθύνη για αγορανομικές παραβάσεις και ερμηνεί...
2,2,**Παραβιάσεις Διαδικαστικών Εγγυήσεων στην Ποι...
3,3,**Καθορισμός Αρμοδιότητας σε Εγκλήσεις κατά Δι...
4,4,**Απαράδεκτη άσκηση αίτησης αναίρεσης κατά πρω...


,cluster,random_title
0,0,**Αναίρεση Ποινικών Αποφάσεων**
1,1,Ζητήματα Ποινικής Δικονομίας και Εφαρμογής Ποι...
2,2,Ζητήματα Απόλυτης Ακυρότητας και Αιτιολογίας σ...
3,3,**Κανονισμός Αρμοδιότητας Δικαστικών και Εισαγ...
4,4,**Δικαιώματα άσκησης αναίρεσης κατά βουλευμάτω...
